In [1]:
from gradient_relevance_score import DistilBertAttributor

In [2]:
attributor = DistilBertAttributor(model_path="./results/distilbert/checkpoint-170")

Using mps device


In [3]:
text = "Stephen J. Gordon (born 4 September 1986) is a chess grandmaster from Oldham, Greater Manchester, England. In September 2004 he took a break from his A-level studies at The Blue Coat School, Oldham to compete in the thirteenth Monarch Assurance Isle of Man International. In 2005, while still a FIDE Master, he finished 6th in the British Championships ahead of a Grandmaster and several International Masters. At the EU Individual Open Chess Championship held at Liverpool in 2006, he led the tournament after eight rounds and finished a very creditable (joint) second, a half point behind winner Nigel Short and level with Luke McShane among others. Probably his best result to date however, was second place in the 2007 British Championship, narrowly losing his share of the lead in the final round. In previous rounds, he defeated both tournament victor Jacob Aagaard and previous champion Jonathan Rowson. By 2008, his rating had reached grandmaster level, although the title itself had not yet been secured. At the British Championship in Liverpool, he almost repeated his performance of the previous year, by taking a share of third place. He was the British under-21 Champion each consecutive year between 2005 and 2008. He became a grandmaster on 1 August 2009. He has been one of the co-presenters of the chess podcast The Full English Breakfast since its inaugural show in October 2010."

In [4]:
target, important_tokens = attributor.compute_attributions(text, merge_scores=False)

In [5]:
def mark_important_tokens(_text, _important_tokens):
    marked_text = _text.replace("[","").replace("]","").lower()
    for token, _ in _important_tokens[:10]:
        marked_text = marked_text.replace(" {} ".format(token), " [{}] ".format(token))
    return marked_text

text_with_attributions = mark_important_tokens(text, important_tokens)

In [6]:
import requests, json

OPENAI_API_KEY = "EMPTY"
with open(".openai_key.txt") as f:
    OPENAI_API_KEY = f.read().strip()
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-3.5-turbo"  # Change to "gpt-4" if needed

HEADERS = {
    "Authorization": f"Bearer {OPENAI_API_KEY}",
    "Content-Type": "application/json"
}

def _send_request(messages, api=API_URL, headers=HEADERS, model=MODEL):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.0 
    }

    response = requests.post(api, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    reply = data["choices"][0]["message"]["content"]
    return reply

In [7]:
prompt = "Explain why the following text is {}anonymized. Words in square brackets [] are important words for the classification of this text. This is the text: ".format("not " if target == 0 else "")

def build_messages(prompt_appendix, text_):
    return [{"role": "user", "content": prompt_appendix + text_}]

In [8]:
_send_request(build_messages(prompt, text_with_attributions))

'The text is not anonymized because it contains specific and unique information about a person named Stephen J. Gordon, including his full name, date of birth, place of birth, achievements in chess, and involvement in a chess podcast. These details are not generalized or obscured in any way, making it easy to identify the individual being discussed.'

In [9]:
from read_jsonl import read_jsonl

eval_df = read_jsonl("DB-bio/combined_val_and_val_sft_anonymized.jsonl")

In [37]:
def get_natural_language_explanation(row):
    print(row.name)
    true_label = row["label"]
    _text = row["text"]
    _target, _important_tokens = attributor.compute_attributions(_text, merge_scores=False)
    _text = mark_important_tokens(_text, _important_tokens)
    _prompt = "Explain why the following text is classified as {}anonymized. Words in square brackets [] are important words for the classification of this text. This is the text: ".format("not " if _target == 0 else "")
    explanation = _send_request(build_messages(_prompt, _text))
    return explanation

In [38]:
len(eval_df)

486

In [39]:
eval_df["explanation"] = eval_df.apply(get_natural_language_explanation,axis=1)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [40]:
eval_df.to_csv("DB-bio/eval_df_with_explations.csv", index=False)

In [41]:
eval_df.head(1)["explanation"]

0    This text is classified as not anonymized beca...
Name: explanation, dtype: object

In [42]:
eval_df.tail(1)["explanation"]

485    This text is classified as anonymized because ...
Name: explanation, dtype: object